# CSV ingest — PVSyst only

Finds the PVSyst CSVs new in `Files/Raw PVSyst1/{plant}/`. 
It parses them with a 10-line header (`PVSYST x.y.z; Project; Geographical Site; ...; date`),
and uploads them in `raw_pvsyst_<plant>` y `proc_pvsyst`.

**SCADA is not consumed by a csv** — `scada_ingest.ipynb`
reads this directly from a processed table in OneLake.

**Idempotent**


In [23]:
plant_filter   = None     
force_reingest = False    # rebuild from scratch (only dev)

StatementMeta(, b084507c-b999-457b-94b2-dbcfbefe15e6, 25, Finished, Available, Finished, False)

In [35]:
import re
import json
import notebookutils
from io import StringIO
from pathlib import Path
from datetime import datetime
import pandas as pd
from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable

# ---------------------------------------------------------
# 1. (Paths for Python: openpyxl, pandas, pathlib)
# ---------------------------------------------------------
LOCAL_FILES = '/lakehouse/default/Files'
CONFIG_XLSX = f'{LOCAL_FILES}/Raw PVSyst1/CONFIG/plants_config.xlsx'

# ---------------------------------------------------------
# 2. (Paths for notebookutils and Spark)
# ---------------------------------------------------------
ONELAKE_FILES      = 'Files'
ONELAKE_RAW_ROOT   = f'{ONELAKE_FILES}/Raw PVSyst1'
ONELAKE_OUTPUT_DIR = f'{ONELAKE_FILES}/output'

ONELAKE_EXCEL_PATH = "Files/Raw PVSyst1/CONFIG/plants_config.xlsx"
LOCAL_TEMP_PATH = "file:///tmp/plants_config.xlsx"
REAL_LOCAL_PATH = "/tmp/plants_config.xlsx"

StatementMeta(, b084507c-b999-457b-94b2-dbcfbefe15e6, 37, Finished, Available, Finished, False)

## 1 · Discover pending CSVs

In [31]:
def discover_csvs(root: str, plant_filter: str | None):
    candidates = []
    for pf in notebookutils.fs.ls(root):
        if not pf.isDir:
            continue
        folder_plant = pf.name.upper()
        if plant_filter and folder_plant != plant_filter.upper():
            continue
        for f in notebookutils.fs.ls(pf.path):
            if not f.name.lower().endswith('.csv'):
                continue
            candidates.append({
                'plant_code': folder_plant,
                'file_kind':  'pvsyst',
                'full_path':  f.path,
                'filename':   f.name,
                'file_size':  f.size,
            })
    return candidates

candidates = discover_csvs(ONELAKE_RAW_ROOT, plant_filter)
print(f'Encontrados {len(candidates)} CSV de PVSyst en SharePoint')


StatementMeta(, b084507c-b999-457b-94b2-dbcfbefe15e6, 33, Finished, Available, Finished, False)

Encontrados 2 CSV de PVSyst en SharePoint


In [55]:
if force_reingest:
    to_ingest = candidates
    print(f'force_reingest=True → procesando los {len(to_ingest)} candidatos')
else:
    already_df = spark.sql("SELECT file_path FROM ingested_files WHERE status = 'ok'")
    already = {r['file_path'] for r in already_df.collect()}
    to_ingest = [c for c in candidates if c['full_path'] not in already]
    print(f'{len(already)} ya ingeridos · {len(to_ingest)} nuevos por procesar')

for c in to_ingest:
    print(f"  {c['plant_code']} · {c['filename']}")


StatementMeta(, b084507c-b999-457b-94b2-dbcfbefe15e6, 57, Finished, Available, Finished, False)

0 ya ingeridos · 2 nuevos por procesar
  PL2 · Plaza-ex_20260514.csv
  PL2 · Plaza-ex2_20260518.csv


## 2 · Verify plants against `T01_Plants`

In [56]:
ABFS_BASE = "abfss://0c4c3bbe-a6ae-404f-b1b7-7052bacfea11@onelake.dfs.fabric.microsoft.com/ae7e1482-8f15-4b1b-8954-8ee500382179"


def _read_excel_table_from_sharepoint(filepath: str, table_name: str) -> pd.DataFrame:
    """Lee una tabla con nombre de un Excel usando ruta ABFS completa."""
    import openpyxl
    local = f'/tmp/{Path(filepath).name}'
    if not Path(local).exists():
        rel   = filepath.replace('/lakehouse/default/', '')
        abfs  = ABFS_BASE + '/' + rel
        row   = spark.read.format('binaryFile').load(abfs).first()
        with open(local, 'wb') as _f:
            _f.write(row['content'])
    wb = openpyxl.load_workbook(local, data_only=True)
    for sheet in wb.worksheets:
        if table_name in sheet.tables:
            tbl  = sheet.tables[table_name]
            data = sheet[tbl.ref]
            rows = [[cell.value for cell in row] for row in data]
            if not rows:
                return pd.DataFrame()
            return pd.DataFrame(rows[1:], columns=rows[0])
    raise ValueError(f"No se encontró la tabla '{table_name}' en {filepath}")

def get_active_plants_dict():
    """Lee el Excel y devuelve un diccionario {plant_code: prefijo_en_mayusculas}"""
    plants_df = _read_excel_table_from_sharepoint(CONFIG_XLSX, 'T01_PLANTS')

    required_cols = ['plant_code', 'pvsyst_file', 'pr_execution']
    for col in required_cols:
        if col not in plants_df.columns:
            raise KeyError(f"Falta la columna '{col}' en T01_PLANTS.")

    # Filtro flexible: acepta "true", "verdadero", "1", "activo", "yes" incluso con símbolos extra
    valores_limpios = plants_df['pr_execution'].astype(str).str.strip().str.lower()
    is_active = valores_limpios.str.contains('true|verdadero|1|si|yes|v|t|activo', regex=True, na=False)

    known = plants_df[is_active].dropna(subset=['plant_code', 'pvsyst_file']).copy()

    known['plant_code'] = known['plant_code'].astype(str).str.strip().str.upper()
    known['pvsyst_file'] = known['pvsyst_file'].astype(str).str.strip().str.upper()

    return dict(zip(known['plant_code'], known['pvsyst_file']))


# 1. Traernos el diccionario de las plantas oficiales
active_plants = get_active_plants_dict()

# 2. Filtrar y re-mapear to_ingest
final_to_ingest = []
for c in to_ingest:
    folder_name = str(c['plant_code']).strip().upper()
    filename = str(c['filename']).strip().upper()

    matched_real_code = None
    for real_pcode, prefix in active_plants.items():
        # Tras el prefijo solo puede venir '_' (fecha) o '.' (extensión)
        if filename.startswith(prefix + '_') or filename.startswith(prefix + '.'):
            matched_real_code = real_pcode
            break

    if matched_real_code:
        c['plant_code'] = matched_real_code
        final_to_ingest.append(c)
    else:
        print(f"  [DEBUG] Descartado: '{filename}' en carpeta '{folder_name}'. Su prefijo no existe o no está activo en T01_PLANTS.")

to_ingest = final_to_ingest
print(f'\nPlantas listas para ingerir tras el filtro: {sorted(list(set(c["plant_code"] for c in to_ingest)))}')

StatementMeta(, b084507c-b999-457b-94b2-dbcfbefe15e6, 58, Finished, Available, Finished, False)

PermissionError: [Errno 13] Permission denied: '/lakehouse/default/Files/Raw PVSyst1/CONFIG/plants_config.xlsx'

## 3 · Helpers de parsing y persistencia

In [53]:
def _read_csv_text(path: str) -> str:
    try:
        with open(path, 'r', encoding='utf-8-sig', errors='replace') as f:
            return f.read()
    except Exception:
        return notebookutils.fs.head(path, 50_000_000)


def read_pvsyst_csv(path: str) -> pd.DataFrame:
    """
    Detectamos la línea cuya primera columna == 'date', saltamos la fila
    siguiente (unidades) y leemos lo demás como CSV separado por ';'.
    """
    raw = _read_csv_text(path)
    lines = raw.splitlines()
    header_idx = None
    for i, line in enumerate(lines[:40]):
        first = line.split(';')[0].strip().lower()
        if first == 'date':
            header_idx = i
            break
    if header_idx is None:
        raise ValueError(f'No encuentro la cabecera "date" en {path}')
    body = '\n'.join([lines[header_idx]] + lines[header_idx + 2:])
    df = pd.read_csv(StringIO(body), sep=';', decimal='.', low_memory=False)
    # PVSyst exporta siempre DD/MM/YY HH:MM; especificamos el formato para
    # evitar el fallback lento a dateutil y garantizar el parseo correcto.
    df[df.columns[0]] = pd.to_datetime(
        df[df.columns[0]], format='%d/%m/%y %H:%M', errors='coerce'
    )
    return df


_PVSYST_RENAME = {
    'date':    'Date',
    'globhor': 'GHI',
    'diffhor': 'DHI',
    't_amb':   'T_AMB',
    'tamb':    'T_AMB',
    'windvel': 'WS',
    'globinc': 'POA',
    'globeff': 'POA',
    'alb_gl':  'ALB',
    'albedo':  'ALB',
    'tarray':  'T_ARRAY',
    'e_grid':  'E_Grid',
    'eoutinv': 'E_Grid',
}


def normalize_pvsyst(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    # Si el CSV contiene GlobInc y GlobEff, conservar solo GlobEff (irradiancia efectiva
    # tras pérdidas ópticas) para evitar columna POA duplicada al renombrar.
    cols_lower = {c.lower().strip(): c for c in df.columns}
    if 'globinc' in cols_lower and 'globeff' in cols_lower:
        df = df.drop(columns=[cols_lower['globinc']])
    rename = {}
    for c in df.columns:
        cl = str(c).lower().strip()
        cl_short = cl.split(' ')[0]
        target = _PVSYST_RENAME.get(cl, _PVSYST_RENAME.get(cl_short))
        if target:
            rename[c] = target
    df = df.rename(columns=rename)
    if 'Date' not in df.columns:
        df = df.rename(columns={df.columns[0]: 'Date'})
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%y %H:%M', errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
    return df


def detect_resolution(ts: pd.Series) -> pd.Timedelta:
    s = pd.to_datetime(ts, errors='coerce').dropna().sort_values()
    diffs = s.diff().dropna()
    return diffs.mode().iloc[0] if not diffs.empty else pd.Timedelta(0)


def write_raw_pvsyst(df_raw: pd.DataFrame, plant: str, source_file: str) -> int:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

    table = 'raw_pvsyst'
    df = df_raw.copy()
    date_col = df.columns[0]

    df['plant_code'] = plant
    df['source_file'] = source_file
    df['ingested_at'] = datetime.utcnow()

    for c in df.columns:
        if c not in (date_col, 'plant_code', 'source_file', 'ingested_at'):
            df[c] = df[c].astype(str)

    sdf = spark.createDataFrame(df)

    if spark.catalog.tableExists(table):
        target = DeltaTable.forName(spark, table)
        target.alias('t').merge(
            sdf.alias('s'),
            f"t.plant_code = s.plant_code AND t.`{date_col}` = s.`{date_col}` AND t.source_file = s.source_file"
        ).whenNotMatchedInsertAll().execute()
    else:
        sdf.write.format('delta') \
           .option("mergeSchema", "true") \
           .partitionBy('plant_code') \
           .mode('append') \
           .saveAsTable(table)

    return len(df)


def write_proc_pvsyst(df_clean: pd.DataFrame, plant: str, source_file: str):
    df = df_clean.copy()
    df['plant_code']  = plant
    df['source_file'] = source_file
    df['ingested_at'] = datetime.utcnow()
    canonical = {'plant_code', 'Date', 'GHI', 'T_AMB', 'ALB', 'POA', 'E_Grid',
                 'source_file', 'ingested_at'}
    drop = [c for c in df.columns if c not in canonical]
    if drop:
        df = df.drop(columns=drop)
    for c in ('GHI', 'T_AMB', 'ALB', 'POA', 'E_Grid'):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        else:
            df[c] = float('nan')
    sdf = spark.createDataFrame(df)
    target = DeltaTable.forName(spark, 'proc_pvsyst')
    target.alias('t').merge(
        sdf.alias('s'),
        't.plant_code = s.plant_code AND t.Date = s.Date'
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


StatementMeta(, b084507c-b999-457b-94b2-dbcfbefe15e6, 55, Finished, Available, Finished, False)

## 4 · Loop principal — un archivo a la vez

In [ ]:
results = []
for c in to_ingest:
    plant = c['plant_code']; path = c['full_path']
    print(f"\n→ {plant}: {c['filename']}")
    status = 'ok'; error = None
    period_start = period_end = None; resolution = None; row_count = 0
    try:
        df_raw    = read_pvsyst_csv(path)
        row_count = write_raw_pvsyst(df_raw, plant, c['filename'])
        df_clean  = normalize_pvsyst(df_raw)
        write_proc_pvsyst(df_clean, plant, c['filename'])
        period_start = df_clean['Date'].min()
        period_end   = df_clean['Date'].max()
        resolution   = str(detect_resolution(df_clean['Date']))
        print(f'  raw + proc merged ({row_count:,} filas · {resolution})')
    except Exception as e:
        status = 'failed'; error = str(e)[:500]
        print(f'  FAILED: {error}')

    results.append(Row(
        plant_code=plant, file_path=path, file_kind='pvsyst',
        file_size=c['file_size'], row_count=row_count,
        period_start=period_start, period_end=period_end,
        resolution=resolution, status=status, error=error,
        ingested_at=datetime.utcnow(),
    ))

if results:
    # Convertimos a pandas primero para que None → NaT en columnas timestamp
    # y Spark pueda inferir los tipos sin CANNOT_DETERMINE_TYPE.
    rows_pd = pd.DataFrame([r.asDict() for r in results])
    for col in ('period_start', 'period_end', 'ingested_at'):
        rows_pd[col] = pd.to_datetime(rows_pd[col], errors='coerce')
    spark.createDataFrame(rows_pd).write.mode('append').format('delta').saveAsTable('ingested_files')
    print(f'\nLogged {len(results)} filas en ingested_files')


## 5 · Recalcular `pending_runs`

In [ ]:
spark.sql('''
    CREATE OR REPLACE TEMPORARY VIEW v_pending AS
    WITH scada AS (
        SELECT plant_code, MIN(Date) AS s_start, MAX(Date) AS s_end, COUNT(*) AS s_rows
        FROM proc_scada GROUP BY plant_code
    ),
    pv AS (
        SELECT plant_code, MIN(Date) AS p_start, MAX(Date) AS p_end
        FROM proc_pvsyst GROUP BY plant_code
    )
    SELECT
        s.plant_code,
        CONCAT(s.plant_code, ' · ',
               date_format(GREATEST(s.s_start, p.p_start), 'yyyy-MM-dd'), ' → ',
               date_format(LEAST(s.s_end, p.p_end),       'yyyy-MM-dd')) AS label,
        GREATEST(s.s_start, p.p_start) AS period_start,
        LEAST(s.s_end, p.p_end)        AS period_end,
        s.s_rows AS row_count,
        current_timestamp()            AS ingested_at
    FROM scada s INNER JOIN pv p ON s.plant_code = p.plant_code
    WHERE GREATEST(s.s_start, p.p_start) < LEAST(s.s_end, p.p_end)
''')
spark.sql('''
    MERGE INTO pending_runs t USING v_pending s
    ON t.plant_code = s.plant_code AND t.label = s.label
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
''')
print(f'pending_runs ahora tiene {spark.table("pending_runs").count()} entradas')
spark.table('pending_runs').show(truncate=False)


In [ ]:
summary = {
    'candidates_found': len(candidates),
    'newly_ingested':   sum(1 for r in results if r['status'] == 'ok'),
    'failed':           sum(1 for r in results if r['status'] == 'failed'),
}
print(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary))
